<div align="center">
  <h1> Sports Concierge Agent</h1>
  <h3>Multi-Agent AI System for Personalised Athletic Training Plans</h3>
  <p><b>Google + Kaggle Agentic AI Hackathon</b></p>
  <p><i>Built with Google ADK · Gemini · FastMCP · Human-in-the-Loop · A2A Protocol</i></p>
  <br>
</div>

---
## Notebook Setup

This notebook serves as the **writeup** for the Sports Concierge Agent. It explains the architecture, workflow, and Google ADK + Gemini integration.

If you want to run the code cells interactively, clone the repository first:

```python
# Uncomment and run to set up the environment:
# !git clone https://github.com/YOUR_USERNAME/sports-concierge-agent
# !cd sports-concierge-agent && pip install -r requirements.txt
# !cd sports-concierge-agent
```

**Video demo:** [View the 6-minute walkthrough on YouTube](https://youtu.be/lJyiqIE-g_M)

**GitHub repo:** [github.com/YOBO-DANCE/Project-GOOGLE-HACKATHON](https://github.com/YOBO-DANCE/Project-GOOGLE-HACKATHON)

---
## Project Overview

The **Sports Concierge Agent** is a multi-agent AI system that generates hyper-personalised sports training plans for athletes. It uses **Google's Agent Development Kit (ADK)** to orchestrate a pipeline of specialised agents:

| Agent | Role |
|-------|------|
| **Guard Agent** (`agents/guard.py`) | Security gate with Human-in-the-Loop (HITL) — scans user input, emails, and files for threats before allowing plan generation |
| **Coach Agent** (`agents/coach.py`) | Elite training plan generator — uses Gemini (or Ollama fallback) to produce personalised, injury-aware workout routines |

The system integrates with **MCP (Model Context Protocol)** tools to gather real-world context — emails from coaches, drive files from training folders, and security scans of retrieved documents — before generating a plan.

---
## Architecture

### High-Level Flow

> Note: The diagram below uses Mermaid syntax. If it doesn't render in this notebook, view the README.md on GitHub for the rendered version.

```text
  ┌─────────────────────────────────────────────────────────────────┐
  │                    USER LAYER                                  │
  │  ┌─────────────────────────────────────────────────────────┐   │
  │  │          Streamlit Dashboard (app.py)                   │   │
  │  │  1. Input: sport, level, injury history                │   │
  │  │  4. HITL Approval (if risk detected)                   │   │
  │  │  6. Display training plan + download + audit trail     │   │
  │  └─────────────────────────────────────────────────────────┘   │
  └─────────────────────────────────────────────────────────────────┘
                              │
              ┌───────────────┴───────────────┐
              ▼                               ▼
  ┌─────────────────────────┐    ┌─────────────────────────┐
  │    GUARD AGENT          │    │    COACH AGENT           │
  │  (agents/guard.py)      │    │  (agents/coach.py)       │
  │  Security analysis      │    │  Training plan gen.      │
  │  + HITL gate            │    │  (Gemini 2.0 Flash)      │
  └─────────────────────────┘    └─────────────────────────┘
              │                              │
              ▼                              ▼
  ┌─────────────────────────┐    ┌─────────────────────────┐
  │  MCP TOOLS              │    │   INFERENCE             │
  │  - get_email_context    │    │  Gemini 2.0 Flash (1°) │
  │  - get_drive_files      │    │  Ollama llama3.2 (2°)  │
  │  - scan_file_security   │    │  Baseline (3°)          │
  └─────────────────────────┘    └─────────────────────────┘
              │
              ▼
  ┌─────────────────────────┐
  │  SECURITY LAYER          │
  │  - YARA pattern matching │
  │  - MIME heuristics       │
  │  - Strict Mode sanitizer │
  └─────────────────────────┘
```

---
## Workflow: 5-Step Pipeline

The workflow is a **server-side-enforced state machine** that prevents skipping security checks:

| Step | Component | Description |
|------|-----------|-------------|
| **1. Input** | Streamlit UI | Athlete enters sport, skill level, injury history |
| **2. Security Check** | Guard Agent + MCP Tools | Gathers emails, drive files; scans for threats via YARA + MIME heuristics |
| **3. Human Approval** | HITL Gate | If risk detected (score >= 30), workflow pauses — human must approve |
| **4. Plan Generation** | Coach Agent + Gemini | Structured JSON payload -> Gemini (or Ollama) -> Markdown training plan |
| **5. Results** | Streamlit UI | Display, download, audit trail |

### Three Layers of Security Enforcement

**Layer 1 — Workflow state machine** (`app.py`): Server-side validation prevents jumping past security checks by manipulating session state.

**Layer 2 — ADK security token** (`sports-concierge/app/agent.py`): The `_generate_training_plan` ADK tool **refuses** to run unless `_check_security` was called first and returned a "proceed" verdict. This is enforced at the Python code level, not just a system prompt:

```python
# Module-level security token — enforces HITL gate at the code level
_security_token: dict | None = None

def _check_security(user_input: str) -> str:
    global _security_token
    decision = guard.process_text(user_input)
    if decision.get("action") == "proceed":
        _security_token = {"status": "approved", "decision": decision}
    else:
        _security_token = None  # clears any previous token
    return json.dumps(decision)

def _generate_training_plan(user_context_json: str) -> str:
    global _security_token
    # Code-level lock — cannot be bypassed by prompt injection
    if _security_token is None or _security_token.get("status") != "approved":
        return "ERROR: Security gate not passed. Call _check_security first."
    # ... proceed with generation ...
    _security_token = None  # require fresh check for each generation
```

**Layer 3 — Input sanitization** (`GuardAgent.sanitize_input`): Strict Mode strips shell injection, XSS, SQL injection, and path traversal before any content reaches the LLM.

---
## Google ADK Integration

This project makes extensive use of **Google's Agent Development Kit (ADK)** — the framework for building production-grade multi-agent systems.

### ADK Agent Definition (`sports-concierge/app/agent.py`)

```python
from google.adk.agents import Agent
from google.adk.apps import App
from google.adk.models import Gemini

root_agent = Agent(
    name="sports_concierge",
    model=Gemini(
        model="gemini-2.0-flash",
        retry_options=types.HttpRetryOptions(attempts=3),
    ),
    instruction=_SPORTS_CONCIERGE_INSTRUCTION,
    tools=[
        _gather_email_context,   # MCP email retrieval
        _gather_drive_files,     # MCP drive file listing
        _scan_file,              # YARA + MIME security scan
        _check_security,         # Guard Agent HITL gate
        _sanitize_text,          # Strict Mode input cleaning
        _generate_training_plan, # Coach Agent plan generation
    ],
)

app = App(root_agent=root_agent, name="sports-concierge")
```

### Key ADK Features Used

| Feature | Usage |
|---------|-------|
| **Gemini Model** | `Gemini(model="gemini-2.0-flash")` with retry options |
| **Tool Functions** | 6 Python functions wrapped as ADK tools with auto-generated JSON Schema |
| **Multi-Agent Topology** | Root agent orchestrates Guard + Coach agents |
| **A2A Protocol** | Agent-to-Agent communication endpoints via `app_utils/a2a.py` |
| **Vertex AI Deployment** | Terraform configs for Agent Engine (auto-scaling, 1-10 instances) |
| **Telemetry** | OpenTelemetry + Cloud Trace + BigQuery logging |

---
## Gemini API Usage

The Coach Agent uses **Gemini 2.0 Flash** (via the `google-genai` SDK) as its primary LLM provider:

```python
from google import genai as genai_sdk

client = genai_sdk.Client(api_key=os.environ["GEMINI_API_KEY"])
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=user_message,
    config=genai_types.GenerateContentConfig(
        system_instruction=elite_coaching_prompt,
        temperature=0.7,
        max_output_tokens=4096,
    ),
)
```

### Why Gemini?

- **Speed**: Gemini 2.0 Flash provides sub-second response times for plan generation
- **Quality**: The elite coaching system prompt enforces structured Markdown output with tables, recovery phases, and injury-specific protocols
- **Context window**: Adequate for the structured JSON payload (goals, injury history, emails, drive files, calendar constraints)

### Fallback Chain

The system auto-detects the best available provider:

1. **Gemini** (cloud) — if `GEMINI_API_KEY` or `GOOGLE_API_KEY` is set (recommended)
2. **Ollama** (local) — if Ollama is running (`llama3.2:3b`)
3. **Baseline** — built-in Couch-to-5K plan if neither is available (the demo never crashes)

---
## Security & Responsible AI

Security is treated as a **first-class citizen** throughout the system. This is a key differentiator for the hackathon — demonstrating Responsible AI practices.

### Guard Agent — Text Analysis

The `GuardAgent.process_text()` method uses **scored keyword detection** to analyse user input:

| Threat Pattern | Score Contribution |
|----------------|--------------------|
| Shell commands (rm, del, format) | 30 |
| Sudo / privilege escalation | 20 |
| Remote downloads (wget, curl) | 25 |
| Script interpreters (bash, python) | 15 |
| SQL injection (DROP TABLE, UNION SELECT) | 40 |
| XSS (<script>, javascript:, onload) | 40 |
| Path traversal (/etc/passwd, ../) | 40 |
| Code execution (eval, exec, os.system) | 35 |

**Score >= 50 -> HALT** (requires human approval)

### YARA + MIME File Scanning

The `SecurityScanner` class (in `security/sandbox.py`) performs two checks on every file:

1. **MIME-type heuristics**: Rejects executable formats (.sh, .exe, .bat, .msi, .jar)
2. **YARA rules**: Detects suspicious code patterns (eval(, base64_decode, Invoke-WebRequest)

### Strict Mode Input Sanitisation

`GuardAgent.sanitize_input()` normalises input (removes null bytes, collapses whitespace, normalises Unicode) then strips:
- Shell injection -> [REDACTED]
- HTML/script tags -> [REDACTED]
- SQL injection -> [REDACTED]
- Prompt injection attempts -> [REDACTED]

### Audit Trail

Every security decision is logged to a **Supabase `security_logs`** table, creating an immutable, queryable audit trail:

| Column | Example |
|--------|---------|
| `created_at` | 2026-07-07T14:30:00Z |
| `filename` | Marathon_Plan.pdf |
| `status` | safe |
| `score` | 0 |
| `reason` | MIME: application/pdf | YARA: no matches |

---
## Context-Aware Plan Generation

The system doesn't just take a sport and level — it builds a **structured context payload** from multiple sources:

```json
{
  "user_goals": "Train in basketball at Intermediate level | ...",
  "training_level": "Intermediate",
  "injury_history": "Achilles tendinopathy (right) | [Physio] Continue eccentric heel-drop exercises...",
  "calendar_constraints": "Team practice Saturday 6 AM...",
  "email_context": "From: coach@example.com ...",
  "drive_files": [
    {"name": "Marathon_Plan.pdf", "type": "application/pdf"}
  ]
}
```

### Injury-Aware Prompting

The app automatically extracts injury keywords from emails:

```python
injury_keywords = ["achilles", "tendon", "strain", "sprain", "fracture",
                  "concussion", "knee", "hamstring", "recovery", "rehab"]
```

If a physio email mentions "Achilles", the Coach Agent receives it in the `injury_history` field and the system prompt **requires** specific exercise protocols (e.g., eccentric heel-drop exercises) in the generated plan.

---
## Project Structure

```
sports-concierge-agent/
├── app.py                          # Streamlit dashboard (orchestrator)
├── agents/
│   ├── guard.py                    # Guard Agent - security + HITL
│   └── coach.py                    # Coach Agent - Gemini/Ollama generation
├── tools/
│   └── server_mcp.py               # FastMCP server (email, drive, scan)
├── security/
│   └── sandbox.py                  # YARA + MIME scanner
├── database/
│   └── manager.py                  # Supabase audit trail
├── Dockerfile                      # Streamlit container
├── docker-compose.yml              # One-command demo (Streamlit + Ollama)
├── DEMO-SCRIPT.md                  # 3-minute judge walkthrough
└── sports-concierge/               # Google ADK project (Vertex AI deployable)
    ├── app/
    │   ├── agent.py                # ADK multi-agent definition (Gemini)
    │   ├── fast_api_app.py         # FastAPI server with A2A
    │   └── app_utils/             # A2A, telemetry, services
    ├── deployment/terraform/       # GCP infrastructure
    └── Dockerfile                  # ADK FastAPI container
```

---
## Quick Start

### Option 1: One-Command Demo (Recommended for Judges)

```bash
docker compose up
# Then open http://localhost:8501
```

### Option 2: Local Development

```bash
pip install -r requirements.txt
cp .env.example .env
# Edit .env and set: GEMINI_API_KEY=your-key
streamlit run app.py
```

### Option 3: Vertex AI Deployment

```bash
./scripts/sync-deps.sh
cd sports-concierge
agents-cli deploy
```

In [ ]:
# Verify all core imports work
# Note: This requires the repo to be cloned and dependencies installed
import sys
sys.path.insert(0, '.')

print("Verifying Sports Concierge Agent imports...")
print("=" * 50)

try:
    from agents.guard import GuardAgent
    print("  [OK] agents.guard")
except Exception as e:
    print(f"  [FAIL] agents.guard: {e}")

try:
    from agents.coach import CoachAgent
    print("  [OK] agents.coach")
except Exception as e:
    print(f"  [FAIL] agents.coach: {e}")

try:
    from tools.server_mcp import get_email_context_sync
    print("  [OK] tools.server_mcp")
except Exception as e:
    print(f"  [FAIL] tools.server_mcp: {e}")

try:
    from security.sandbox import SecurityScanner
    print("  [OK] security.sandbox")
except Exception as e:
    print(f"  [FAIL] security.sandbox: {e}")

try:
    from database.manager import db_manager
    print("  [OK] database.manager")
except Exception as e:
    print(f"  [FAIL] database.manager: {e}")

print("=" * 50)
print("All systems nominal.")

---
##  Live Security Demo (Run in Notebook)

The Guard Agent can detect and neutralise threats in real-time. Try the examples below:

In [ ]:
from agents.guard import GuardAgent

guard = GuardAgent()

# Test 1: Safe input
safe = "I want to train for a marathon. I have mild shin splints."
result = guard.process_text(safe)
print(f"Safe input -> {result['action']} (score={result.get('score', '?')})")

# Test 2: Malicious input (SQL injection)
malicious = "Help me with sprinting; DROP TABLE users; --"
result = guard.process_text(malicious)
print(f"Malicious input -> {result['action']} (score={result.get('score', '?')})")

# Test 3: Strict Mode sanitisation
dirty = "Run: rm -rf / && <script>alert('xss')</script>"
clean = GuardAgent.sanitize_input(dirty)
print(f"Sanitised: {clean}")

---
## Coach Agent Demo

The Coach Agent generates training plans. Without a Gemini API key, it falls back to a **built-in baseline plan** (Couch-to-5K). With `GEMINI_API_KEY` set in `.env`, it uses Gemini 2.0 Flash for fully personalised plans.

Run the cell below to see the baseline fallback in action. For Gemini-generated plans, run the Streamlit dashboard instead.

In [ ]:
from agents.coach import CoachAgent

coach = CoachAgent()
# Empty context triggers the built-in baseline plan
plan = coach.generate_training_plan(user_context="")
print(plan[:600] + "...\n[truncated]")

---
## Evaluation & Quality Assurance

The project includes an evaluation framework (via `agents-cli eval`) to measure agent quality:

| Metric | Description | Tool |
|--------|-------------|------|
| **Response Quality** | LLM-as-judge evaluation of training plan relevance | `metrics.py` |
| **Agent Turn Count** | Number of reasoning loops before completion | Built-in |
| **Security Accuracy** | Guard Agent correctly identifies threats without false positives | Manual |
| **Rate Limiting** | Enforces 5-second cooldown, locks out after 5 violations | Server-side |

To run evaluation:

```bash
cd sports-concierge
agents-cli eval generate
agents-cli eval grade
```

---
## Google Cloud Stack Summary

| Service | Usage | Status |
|---------|-------|--------|
| **Google ADK** | Multi-agent framework (Guard + Coach agents) | Core |
| **Gemini API** | LLM inference for training plan generation | Core |
| **A2A Protocol** | Agent-to-Agent communication endpoints | Integrated |
| **Vertex AI Agent Engine** | Production deployment target | Terraform configs |
| **Cloud Trace** | Distributed tracing via OpenTelemetry | Configured |
| **Cloud Logging** | Structured logging and feedback collection | Configured |
| **BigQuery** | Telemetry log analysis | Schema provided |
| **Cloud Storage** | Artifact persistence (logs bucket) | Terraform config |
| **Cloud Run** | Containerised ADK server deployment | Dockerfile ready |

---
## Key Differentiators for Judges

| What We Do | Why It Matters |
|------------|----------------|
| **Multi-Agent with HITL** | Not just a prompt — two specialised agents with a code-enforced security gate |
| **Context-Aware Plans** | Extracts injury data from emails, schedules from calendars — not a generic template |
| **Immutable Audit Trail** | Every security decision is logged to Supabase for compliance (SOC 2, ISO 27001) |
| **Fallback Resilient** | Gemini -> Ollama -> Baseline — the demo never crashes |
| **Harness Engineering** | MCP tools, rate limiting, XSS sanitisation, server-side workflow validation |
| **Google Cloud Ready** | ADK + Gemini + Vertex AI + A2A — deployable with one command |

### Submission Checklist

- [x] GitHub repo with full source code
- [x] README with architecture diagram and setup guide
- [x] One-command demo (docker compose up)
- [x] DEMO-SCRIPT.md with 3-minute judge walkthrough
- [x] Kaggle writeup notebook (this file)
- [ ] Video demo (upload to YouTube, link in setup cell above)
- [ ] Update GitHub URL and video ID placeholders

---

<div align="center">
  <p><b>Sports Concierge Agent</b></p>
  <p>Google + Kaggle Agentic AI Hackathon 2026</p>
  <p>Built with Google ADK, Gemini, FastMCP, and Streamlit</p>
</div>